# 🚀 YOLOv5-C2f 口罩检测
> 自定义 C2f 模块 + Face Mask Detection 数据集 | T4 GPU

**Shift+Enter 逐个运行，不要一次性全部运行**

## 1. 检查 GPU + 挂载 Google Drive

In [ ]:
# 检查 GPU 是否可用（必须是 T4）
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库 + 安装依赖

In [ ]:
# 克隆你的仓库（包含 c2f 修改）
!git clone https://github.com/qianz7884-blip/yolo.git
%cd yolo

In [ ]:
# 安装依赖（requirements.txt 已修复）
!pip install -q -r requirements.txt
!pip install -q kagglehub
print('✅ 依赖安装完成')

## 3. 下载数据集（Face Mask Detection）

In [ ]:
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path
import kagglehub

print('📥 下载 Face Mask Detection 数据集...')
src = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))

# 创建 YOLO 目录结构（与 data/mask.yaml 的 path 对应）
dst = Path('./datasets/mask')
for split in ['train', 'val']:
    (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}

# 解析 XML 标注 → YOLO 格式
all_data = []
for xml_path in sorted((src / 'annotations').glob('*.xml')):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)

    objs = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP:
            continue
        bbox = obj.find('bndbox')
        xc = ((float(bbox.find('xmin').text) + float(bbox.find('xmax').text)) / 2) / w
        yc = ((float(bbox.find('ymin').text) + float(bbox.find('ymax').text)) / 2) / h
        bw = (float(bbox.find('xmax').text) - float(bbox.find('xmin').text)) / w
        bh = (float(bbox.find('ymax').text) - float(bbox.find('ymin').text)) / h
        objs.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

    if objs:  # 只保留有标注的图片
        all_data.append({'filename': filename, 'objects': objs})

# 80/20 分割（固定种子，可复现）
random.seed(42)
random.shuffle(all_data)
n = int(len(all_data) * 0.8)

for split, data in [('train', all_data[:n]), ('val', all_data[n:])]:
    for item in data:
        stem = Path(item['filename']).stem
        img_src = src / 'images' / item['filename']
        if img_src.exists():
            shutil.copy2(img_src, dst / 'images' / split / item['filename'])
        lbl_path = dst / 'labels' / split / f'{stem}.txt'
        lbl_path.write_text('\n'.join(item['objects']) + ('\n' if item['objects'] else ''))

print(f'✅ 训练集 {n} 张,  验证集 {len(all_data) - n} 张')

## 4. 训练（yolov5s-c2f）

In [ ]:
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data data/mask.yaml \
    --weights yolov5s.pt \
    --cfg models/yolov5s-c2f.yaml \
    --project /content/drive/MyDrive/yolov5-mask \
    --name exp-c2f \
    --workers 2

## 5. 查看训练结果

In [ ]:
from IPython.display import Image, display

exp = '/content/drive/MyDrive/yolov5-mask/exp-c2f'

print('📈 训练曲线')
display(Image(filename=f'{exp}/results.png'))

print('📊 混淆矩阵')
display(Image(filename=f'{exp}/confusion_matrix.png'))

print('🔍 验证样张')
display(Image(filename=f'{exp}/val_batch0_pred.jpg'))

## 6. 推理检测（上传图片）

In [ ]:
from google.colab import files
import glob

uploaded = files.upload()
for fname in uploaded.keys():
    !python detect.py \
        --weights /content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt \
        --source {fname} \
        --conf 0.25
    result = glob.glob(f'/content/yolo/runs/detect/*/{fname}')
    if result:
        display(Image(filename=result[0]))

## 7. 下载模型

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt')